# Faz 3 — YOLOv8 Baseline (Fold 0)

**Amaç:** Faz 2'de hazırlanan YOLO veri seti üzerinde Ultralytics YOLOv8 baseline'ı eğitmek ve **F1 @ IoU top-5 ortalama** metriğiyle değerlendirmek (Koç 2024 standardı).

**Adımlar**
1. Ortam + GPU kontrolü
2. Eğitim (hızlı baseline: epoch=10; tam: 50)
3. En iyi ağırlıkla val seti inference
4. F1@IoU ∈ {0.1, 0.2, 0.3, 0.4, 0.5} → top-5 ortalama
5. Sınıf bazında precision/recall/F1
6. Görsel kontrol (GT mavi, tahmin kırmızı)
7. Metrik raporu JSON

**Süre tahmini:** T4/P100 + 10 epoch ~30-60 dk; tam 50 epoch ~3-5 saat (Kaggle 30sa/hafta limiti içinde).

## 1. Ortam

In [1]:
!pip install ipywidgets

In [2]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))
YARISMA_DIR = Path(os.environ.get("ABDOMEN_TEST_DIR", DATA_ROOT / "Test Verisi"))

# os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
# os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
# os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
# os.environ["ABDOMEN_TEST_DIR"]     = str(DATA_ROOT / "Yarışma Veri Seti")
# os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
# os.environ["ABDOMEN_OUT_DIR"]      = str(PROJE / "outputs")

sys.path.insert(0, str(CODE))
from src.config import DET_DATA_DIR, CKPT_DIR, SUPER_CLASSES, DEFAULT_DET

fold = 0
fold_dir = DET_DATA_DIR / f"fold{fold}"
dataset_yaml = fold_dir / "dataset.yaml"
print("YOLO veri seti:", fold_dir, "→ var?", fold_dir.exists())
print("dataset.yaml :", dataset_yaml.exists())

YOLO veri seti: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/det_data/fold0 → var? True
dataset.yaml : True


In [3]:
import torch
print(f"PyTorch         : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
elif torch.backends.mps.is_available():
    print("MPS device      : Apple Silicon")
else:
    print("⚠️  GPU yok — Kaggle/Colab GPU runtime önerilir.")

PyTorch         : 2.8.0
CUDA available  : False
MPS device      : Apple Silicon


## 2. Eğitim Konfigürasyonu

In [ ]:
from dataclasses import replace

# DEFAULT_DET artık CT'ye uyarlanmış ayarları içeriyor (src/config.py)
# Yerel MPS için batch/imgsz küçültüldü; Kaggle/Colab'da DEFAULT_DET'i direkt kullan
local_cfg = replace(DEFAULT_DET,
    model="yolov8m.pt",
    img_size=512,
    batch_size=8,       # MPS unified memory sınırı
    epochs=100,
    patience=30,
    # ── Recall artırma ──────────────────────────────────────────────────
    degrees=10.0,       # ±10° rotasyon — CT dilim açı varyasyonu
    erasing=0.1,        # düşürüldü (0.4→0.1) — küçük lezyonları silmemek için
    copy_paste=0.3,     # lezyon kopyalama — nadir sınıf recall artışı
    cls_loss=0.3,       # düşürüldü (0.5→0.3) — daha fazla kutu → yüksek recall
)
print("Eğitim ayarları (local_cfg):")
for k, v in local_cfg.__dict__.items():
    print(f"  {k}: {v}")

## 3. Eğitim

In [ ]:
from src.detection import train_yolo
weights_path = train_yolo(fold=fold, cfg=local_cfg, project=str(PROJE / "outputs" / "runs" / "det"))
print("En iyi ağırlık:", weights_path)

New https://pypi.org/project/ultralytics/8.4.54 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.38 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ramazanpolat/Desktop/datasets/abdomen/outputs/det_data/fold0/dataset.yaml, degrees=0.0, deterministic=False, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0

## 4. Validation Inference

In [ ]:
# Eğitim olmadan direkt test etmek için:
# weights_path = os.path.join("runs", "det", "fold0_yolov8s", "weights", "best.pt")

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from ultralytics import YOLO
from tqdm import tqdm

model = YOLO(str(weights_path))
val_dir = fold_dir / "images" / "val"
val_imgs = sorted(val_dir.glob("*.png"))
print(f"Val PNG sayısı: {len(val_imgs)}")

CONF_TH = 0.05
preds = []
for ip in tqdm(val_imgs, desc="val inference"):
    stem = ip.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    res = model.predict(str(ip), conf=CONF_TH, verbose=False, imgsz=local_cfg.img_size)[0]
    for box, sc, cl in zip(res.boxes.xyxy.cpu().numpy(),
                           res.boxes.conf.cpu().numpy(),
                           res.boxes.cls.cpu().numpy()):
        preds.append({"case": int(case), "image_id": int(image_id), "class": int(cl),
                      "score": float(sc),
                      "x1": float(box[0]), "y1": float(box[1]),
                      "x2": float(box[2]), "y2": float(box[3])})
pred_df = pd.DataFrame(preds)
out = PROJE / "outputs" / "logs" / f"yolo_fold{fold}_val_preds.csv"
out.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(out, index=False)
print(f"Tahmin kaydedildi: {len(pred_df):,} satır → {out}")

## 5. Ground Truth

In [ ]:
val_lbl_dir = fold_dir / "labels" / "val"
gt_rows = []
for lp in val_lbl_dir.glob("*.txt"):
    ip = val_dir / (lp.stem + ".png")
    if not ip.exists(): continue
    W, H = Image.open(ip).size
    stem = lp.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5: continue
        cl = int(p[0]); cx,cy,w,h = map(float, p[1:5])
        gt_rows.append({"case": int(case), "image_id": int(image_id), "class": cl,
                        "x1": (cx-w/2)*W, "y1": (cy-h/2)*H,
                        "x2": (cx+w/2)*W, "y2": (cy+h/2)*H})
gt_df = pd.DataFrame(gt_rows)
print(f"GT BBox: {len(gt_df):,}")
gt_df.head(3)

## 6. F1 @ IoU Değerlendirme

In [ ]:
from src.evaluation import f1_at_iou, top5_f1_mean
result = top5_f1_mean(pred_df, gt_df)
print("Top-5 ortalama F1 :", round(result['top5_mean_f1'], 4))
print("Top-5 eşikleri    :", [(round(t,2), round(f,4)) for t,f in result['top5']])
print("\nTüm eşiklerde:")
for t, f in result['per_threshold']:
    print(f"  IoU={t:.1f} → macro F1={f:.4f}")

In [ ]:
detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)
print(f"Macro F1 @ IoU 0.3: {detail['macro_f1']:.4f}")
print(f"Micro F1 @ IoU 0.3: {detail['micro_f1']:.4f}\n")
rows = []
for cid, m in detail['per_class'].items():
    rows.append({
        "class_id": cid,
        "class_name": SUPER_CLASSES[cid] if cid < len(SUPER_CLASSES) else f"id{cid}",
        "precision": round(m['precision'], 3),
        "recall":    round(m['recall'], 3),
        "f1":        round(m['f1'], 3),
        "TP": m['tp'], "FP": m['fp'], "FN": m['fn'],
    })
cls_table = pd.DataFrame(rows).sort_values("class_id")
cls_table

## 7. Görsel Sağlama

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
random.seed(11)
samples = random.sample(val_imgs, min(4, len(val_imgs)))
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, ip in zip(axes, samples):
    img = np.array(Image.open(ip))
    ax.imshow(img)
    stem = ip.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    case, image_id = int(case), int(image_id)
    g = gt_df[(gt_df['case'] == case) & (gt_df['image_id'] == image_id)]
    for _, r in g.iterrows():
        ax.add_patch(mpatches.Rectangle((r.x1, r.y1), r.x2-r.x1, r.y2-r.y1,
                                        linewidth=2, edgecolor='blue',
                                        facecolor='none', linestyle='--'))
    p = pred_df[(pred_df['case'] == case) & (pred_df['image_id'] == image_id) & (pred_df['score'] >= 0.25)]
    for _, r in p.iterrows():
        ax.add_patch(mpatches.Rectangle((r.x1, r.y1), r.x2-r.x1, r.y2-r.y1,
                                        linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(r.x1, r.y1-3, f"{SUPER_CLASSES[r['class']][:12]} ({r.score:.2f})",
                color='red', fontsize=8, backgroundcolor='white')
    ax.set_title(f"{case}/{image_id}", fontsize=9); ax.axis('off')
plt.suptitle("Mavi=GT, Kırmızı=Tahmin (conf≥0.25)", fontsize=11)
plt.tight_layout(); plt.show()

## 8. Metrik Raporu

In [ ]:
import json
result_path = CODE / "outputs" / "logs" / f"yolo_fold{fold}_metrics.json"
with open(result_path, "w", encoding="utf-8") as f:
    json.dump({
        "fold": fold,
        "config": local_cfg.__dict__,
        "top5_mean_f1": result['top5_mean_f1'],
        "per_threshold": [(round(t,2), round(f,5)) for t,f in result['per_threshold']],
        "per_class_at_iou_0.3": {SUPER_CLASSES[cid]: {k: round(v,4) if isinstance(v,float) else v
                                                       for k,v in m.items()}
                                 for cid, m in detail['per_class'].items()},
    }, f, indent=2, ensure_ascii=False)
print("Metrik raporu:", result_path)
print(f"\n📊 ÖZET — Fold 0 YOLO Baseline:")
print(f"   Top-5 ortalama F1   : {result['top5_mean_f1']:.4f}")
print(f"   Macro F1 @ IoU 0.3  : {detail['macro_f1']:.4f}")
print(f"   Micro F1 @ IoU 0.3  : {detail['micro_f1']:.4f}")

## 9. Faz 3 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| Eğitilmiş ağırlık | `outputs/checkpoints/yolo_fold0.pt` |
| Val tahminleri | `outputs/logs/yolo_fold0_val_preds.csv` |
| Metrik raporu | `outputs/logs/yolo_fold0_metrics.json` |

**Sonraki:** `Faz4_nnDetection_3B_1fold.ipynb`